# Big Data Analytics - Mini Project 2
## Spark Case Study: Flight Delay & Cancellation Analysis (2019-2023)

**Dataset:** US Flight Delay & Cancellation Data (Kaggle - patrickzel/flight-delay-and-cancellation-dataset-2019-2023)

**Size:** 3,000,000 records x 32 columns (~585 MB CSV)

---

### Research Question
**What drives flight delay severity across US airlines and routes, and how do delay causes propagate over time?**

We investigate:
1. **Trend analysis** - Temporal delay patterns (moving averages, cumulative cancellations)
2. **Anomaly detection** - Routes/airlines with statistically abnormal delays
3. **Root-cause attribution** - Which delay category (carrier, weather, NAS, security, late aircraft) dominates per airline

### Why this dataset is suitable for distributed processing
- **3M rows** exceed single-node memory-efficient processing for complex window/join workloads
- **32 mixed-type columns** (timestamps, numerics, categoricals, nullable delay codes) exercise all Catalyst rules
- Natural support for **join optimization** via small lookup tables (airlines, airports)
- Rich **time dimension** enables window functions (moving averages, cumulative sums, ranking)

## 1. Environment Setup

In [1]:
import os, time, shutil
from pathlib import Path

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel

print('PySpark version:', pyspark.__version__)

PySpark version: 4.1.1


In [2]:
# Build Spark session. Local cluster, 4 executors via [4] - adjust for scalability tests.
import sys

# Fix 1: Java 17+ needs security-manager flag, else getSubject throws UnsupportedOperationException
JAVA_OPTS = '-Djava.security.manager=allow'

# Fix 2: On Windows, Python workers crash unless PYSPARK_PYTHON points at the driver's Python
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

os.environ['PYSPARK_SUBMIT_ARGS'] = (
    f'--conf spark.driver.extraJavaOptions={JAVA_OPTS} '
    f'--conf spark.executor.extraJavaOptions={JAVA_OPTS} '
    'pyspark-shell'
)

In [3]:
# Build Spark session. Local cluster, 4 executors via [4] - adjust for scalability tests.
spark = (SparkSession.builder
         .appName('FlightDelayCaseStudy')
         .master('local[4]')
         .config('spark.sql.shuffle.partitions', '200')
         .config('spark.driver.memory', '4g')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.autoBroadcastJoinThreshold', 10 * 1024 * 1024)
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel('WARN')
print('Spark UI:', sc.uiWebUrl)
print('Default parallelism:', sc.defaultParallelism)

Spark UI: http://host.docker.internal:4040
Default parallelism: 4


## 2. Dataset Loading & Schema

We define the schema explicitly - this skips Spark's schema-inference scan and makes the benchmarks fair.

In [4]:
CSV_PATH = 'flights_sample_3m.csv'
PARQUET_PATH = 'flights.parquet'

schema = StructType([
    StructField('FL_DATE', DateType(), True),
    StructField('AIRLINE', StringType(), True),
    StructField('AIRLINE_DOT', StringType(), True),
    StructField('AIRLINE_CODE', StringType(), True),
    StructField('DOT_CODE', IntegerType(), True),
    StructField('FL_NUMBER', IntegerType(), True),
    StructField('ORIGIN', StringType(), True),
    StructField('ORIGIN_CITY', StringType(), True),
    StructField('DEST', StringType(), True),
    StructField('DEST_CITY', StringType(), True),
    StructField('CRS_DEP_TIME', IntegerType(), True),
    StructField('DEP_TIME', DoubleType(), True),
    StructField('DEP_DELAY', DoubleType(), True),
    StructField('TAXI_OUT', DoubleType(), True),
    StructField('WHEELS_OFF', DoubleType(), True),
    StructField('WHEELS_ON', DoubleType(), True),
    StructField('TAXI_IN', DoubleType(), True),
    StructField('CRS_ARR_TIME', IntegerType(), True),
    StructField('ARR_TIME', DoubleType(), True),
    StructField('ARR_DELAY', DoubleType(), True),
    StructField('CANCELLED', DoubleType(), True),
    StructField('CANCELLATION_CODE', StringType(), True),
    StructField('DIVERTED', DoubleType(), True),
    StructField('CRS_ELAPSED_TIME', DoubleType(), True),
    StructField('ELAPSED_TIME', DoubleType(), True),
    StructField('AIR_TIME', DoubleType(), True),
    StructField('DISTANCE', DoubleType(), True),
    StructField('DELAY_DUE_CARRIER', DoubleType(), True),
    StructField('DELAY_DUE_WEATHER', DoubleType(), True),
    StructField('DELAY_DUE_NAS', DoubleType(), True),
    StructField('DELAY_DUE_SECURITY', DoubleType(), True),
    StructField('DELAY_DUE_LATE_AIRCRAFT', DoubleType(), True),
])

flights = (spark.read
           .option('header', 'true')
           .schema(schema)
           .csv(CSV_PATH))

# Derived columns used across multiple queries
flights = (flights
           .withColumn('YEAR', F.year('FL_DATE'))
           .withColumn('MONTH', F.month('FL_DATE'))
           .withColumn('DAY_OF_WEEK', F.dayofweek('FL_DATE')))

flights.printSchema()
print('Total records:', flights.count())

root
 |-- FL_DATE: date (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- AIRLINE_DOT: string (nullable = true)
 |-- AIRLINE_CODE: string (nullable = true)
 |-- DOT_CODE: integer (nullable = true)
 |-- FL_NUMBER: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)

In [5]:
# Register as SQL temp view for the SQL API implementations
flights.createOrReplaceTempView('flights')
spark.sql('SELECT COUNT(*) AS n FROM flights').show()

+-------+
|      n|
+-------+
|3000000|
+-------+



### Build secondary lookup tables (for join demonstrations)

We derive two side tables from the main dataset:
- **airlines_dim** - small table (~20 rows) - ideal for **broadcast join**
- **airport_stats** - medium table (~360 rows) - can demonstrate **sort-merge** when broadcast is disabled

In [6]:
airlines_dim = (flights.select('AIRLINE_CODE', 'AIRLINE', 'DOT_CODE')
                .distinct())
airlines_dim.createOrReplaceTempView('airlines_dim')
print('airlines_dim rows:', airlines_dim.count())

airport_stats = (flights.groupBy('ORIGIN')
                 .agg(F.count('*').alias('ORIGIN_FLIGHT_COUNT'),
                      F.first('ORIGIN_CITY').alias('ORIGIN_CITY')))
airport_stats.createOrReplaceTempView('airport_stats')
print('airport_stats rows:', airport_stats.count())

airlines_dim rows: 18
airport_stats rows: 380


### Benchmark helper

A single utility records execution time and triggers an action so lazy transformations actually execute.

In [7]:
PERF = []  # list of dicts for the final performance comparison table

def time_it(label, api, fn):
    t0 = time.perf_counter()
    result = fn()
    if hasattr(result, 'count'):
        n = result.count()
    else:
        n = int(result)
    elapsed = time.perf_counter() - t0
    PERF.append({'query': label, 'api': api, 'seconds': round(elapsed, 3), 'rows': n})
    print(f'[{label:25s}][{api:10s}] {elapsed:.3f}s  rows={n}')
    return result

---

## 3. Queries (12) - implemented in RDD, DataFrame, and Spark SQL

For each query we:
1. Describe the analytical intent.
2. Implement it **three ways**.
3. Benchmark execution time.
4. Print `.explain(True)` for the DataFrame version - includes **parsed, analyzed, optimized logical, and physical** plans.

In [8]:
# Shared RDD used by every RDD-based query.
flights_rdd = flights.rdd
flights_rdd.cache()

MapPartitionsRDD[39] at javaToPython at DirectMethodHandleAccessor.java:103

### Q1. Filtering with complex conditions

> Find flights in **winter months (Dec/Jan/Feb)** that were **delayed >= 60 minutes at arrival**, **not cancelled**, and had **distance > 500 miles**.

In [9]:
# --- RDD API ---
def q1_rdd():
    return flights_rdd.filter(lambda r: r.MONTH in (12, 1, 2)
                                     and r.ARR_DELAY is not None and r.ARR_DELAY >= 60
                                     and r.CANCELLED == 0.0
                                     and r.DISTANCE is not None and r.DISTANCE > 500)
time_it('Q1_filter', 'RDD', lambda: q1_rdd().count())

# --- DataFrame API ---
def q1_df():
    return (flights
            .filter(F.col('MONTH').isin(12, 1, 2))
            .filter(F.col('ARR_DELAY') >= 60)
            .filter(F.col('CANCELLED') == 0.0)
            .filter(F.col('DISTANCE') > 500))
q1_df_res = time_it('Q1_filter', 'DataFrame', q1_df)

# --- Spark SQL ---
def q1_sql():
    return spark.sql("""
        SELECT * FROM flights
        WHERE MONTH IN (12,1,2)
          AND ARR_DELAY >= 60
          AND CANCELLED = 0.0
          AND DISTANCE > 500
    """)
time_it('Q1_filter', 'SQL', q1_sql)

print('\n=== Q1 DataFrame .explain(True) ===')
q1_df_res.explain(True)

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 2 in stage 18.0 failed 1 times, most recent failure: Lost task 2.0 in stage 18.0 (TID 28) (host.docker.internal executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$an

### Q2. Aggregations (SUM, AVG, COUNT, MAX, MIN)

> Aggregate delay statistics per airline: total flights, cancellation count, avg / min / max arrival delay.

In [ ]:
# --- RDD API ---
def q2_rdd():
    kv = flights_rdd.map(lambda r: (r.AIRLINE_CODE, (
        1,
        float(r.CANCELLED or 0),
        float(r.ARR_DELAY) if r.ARR_DELAY is not None else 0.0,
        1 if r.ARR_DELAY is not None else 0,
        float(r.ARR_DELAY) if r.ARR_DELAY is not None else float('-inf'),
        float(r.ARR_DELAY) if r.ARR_DELAY is not None else float('inf'))))
    reduced = kv.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1], a[2]+b[2], a[3]+b[3], max(a[4], b[4]), min(a[5], b[5])))
    return reduced.map(lambda x: (x[0], x[1][0], x[1][1],
                                  x[1][2]/x[1][3] if x[1][3] else None, x[1][4], x[1][5]))
time_it('Q2_aggregate', 'RDD', lambda: q2_rdd().count())

# --- DataFrame API ---
def q2_df():
    return (flights.groupBy('AIRLINE_CODE')
            .agg(F.count('*').alias('total_flights'),
                 F.sum('CANCELLED').alias('cancellations'),
                 F.avg('ARR_DELAY').alias('avg_arr_delay'),
                 F.max('ARR_DELAY').alias('max_arr_delay'),
                 F.min('ARR_DELAY').alias('min_arr_delay')))
q2_df_res = time_it('Q2_aggregate', 'DataFrame', q2_df)

# --- Spark SQL ---
def q2_sql():
    return spark.sql("""
        SELECT AIRLINE_CODE,
               COUNT(*)          AS total_flights,
               SUM(CANCELLED)    AS cancellations,
               AVG(ARR_DELAY)    AS avg_arr_delay,
               MAX(ARR_DELAY)    AS max_arr_delay,
               MIN(ARR_DELAY)    AS min_arr_delay
        FROM flights
        GROUP BY AIRLINE_CODE
    """)
time_it('Q2_aggregate', 'SQL', q2_sql)

q2_df_res.orderBy(F.desc('total_flights')).show(truncate=False)
print('\n=== Q2 DataFrame .explain(True) ===')
q2_df_res.explain(True)

### Q3. Grouping by multiple attributes

> Average arrival delay per **(airline, origin airport, month)** - useful for hotspot detection.

In [ ]:
# --- RDD API ---
def q3_rdd():
    kv = (flights_rdd
          .filter(lambda r: r.ARR_DELAY is not None)
          .map(lambda r: ((r.AIRLINE_CODE, r.ORIGIN, r.MONTH), (float(r.ARR_DELAY), 1))))
    return (kv.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
              .mapValues(lambda v: v[0]/v[1]))
time_it('Q3_multi_group', 'RDD', lambda: q3_rdd().count())

# --- DataFrame API ---
def q3_df():
    return (flights.groupBy('AIRLINE_CODE', 'ORIGIN', 'MONTH')
            .agg(F.avg('ARR_DELAY').alias('avg_arr_delay'),
                 F.count('*').alias('n')))
q3_df_res = time_it('Q3_multi_group', 'DataFrame', q3_df)

# --- Spark SQL ---
def q3_sql():
    return spark.sql("""
        SELECT AIRLINE_CODE, ORIGIN, MONTH,
               AVG(ARR_DELAY) AS avg_arr_delay,
               COUNT(*)       AS n
        FROM flights
        WHERE ARR_DELAY IS NOT NULL
        GROUP BY AIRLINE_CODE, ORIGIN, MONTH
    """)
time_it('Q3_multi_group', 'SQL', q3_sql)

print('\n=== Q3 DataFrame .explain(True) ===')
q3_df_res.explain(True)

### Q4. Sorting and ranking - Top 20 most-delayed routes

In [ ]:
# --- RDD API ---
def q4_rdd():
    kv = (flights_rdd
          .filter(lambda r: r.ARR_DELAY is not None)
          .map(lambda r: ((r.ORIGIN, r.DEST), (float(r.ARR_DELAY), 1))))
    agg = kv.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])).mapValues(lambda v: v[0]/v[1])
    return agg.takeOrdered(20, key=lambda x: -x[1])
time_it('Q4_top20_routes', 'RDD', lambda: len(q4_rdd()))

# --- DataFrame API ---
def q4_df():
    return (flights.groupBy('ORIGIN', 'DEST')
            .agg(F.avg('ARR_DELAY').alias('avg_delay'),
                 F.count('*').alias('flights'))
            .filter('flights > 100')
            .orderBy(F.desc('avg_delay'))
            .limit(20))
q4_df_res = time_it('Q4_top20_routes', 'DataFrame', q4_df)

# --- Spark SQL ---
def q4_sql():
    return spark.sql("""
        SELECT ORIGIN, DEST,
               AVG(ARR_DELAY) AS avg_delay,
               COUNT(*)       AS flights
        FROM flights
        GROUP BY ORIGIN, DEST
        HAVING COUNT(*) > 100
        ORDER BY avg_delay DESC
        LIMIT 20
    """)
time_it('Q4_top20_routes', 'SQL', q4_sql)

q4_df_res.show(20, truncate=False)
print('\n=== Q4 DataFrame .explain(True) ===')
q4_df_res.explain(True)

### Q5. Window function - 7-day moving average of arrival delay per airline

Demonstrates `ROWS BETWEEN N PRECEDING AND CURRENT ROW` - a key trend-analysis tool.

In [ ]:
# --- RDD API (manual sliding window after groupByKey) ---
def q5_rdd():
    daily = (flights_rdd
             .filter(lambda r: r.ARR_DELAY is not None)
             .map(lambda r: ((r.AIRLINE_CODE, r.FL_DATE), (float(r.ARR_DELAY), 1)))
             .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
             .map(lambda x: (x[0][0], (x[0][1], x[1][0]/x[1][1]))))
    grouped = daily.groupByKey().mapValues(lambda it: sorted(it))
    def rolling(pairs):
        out = []
        for i in range(len(pairs)):
            window = pairs[max(0, i-6):i+1]
            out.append((pairs[i][0], sum(p[1] for p in window)/len(window)))
        return out
    return grouped.flatMapValues(rolling)
time_it('Q5_moving_avg', 'RDD', lambda: q5_rdd().count())

# --- DataFrame API ---
def q5_df():
    daily = (flights.filter(F.col('ARR_DELAY').isNotNull())
             .groupBy('AIRLINE_CODE', 'FL_DATE')
             .agg(F.avg('ARR_DELAY').alias('daily_avg')))
    w = Window.partitionBy('AIRLINE_CODE').orderBy('FL_DATE').rowsBetween(-6, 0)
    return daily.withColumn('ma7', F.avg('daily_avg').over(w))
q5_df_res = time_it('Q5_moving_avg', 'DataFrame', q5_df)

# --- Spark SQL ---
def q5_sql():
    return spark.sql("""
        WITH daily AS (
          SELECT AIRLINE_CODE, FL_DATE, AVG(ARR_DELAY) AS daily_avg
          FROM flights WHERE ARR_DELAY IS NOT NULL
          GROUP BY AIRLINE_CODE, FL_DATE
        )
        SELECT AIRLINE_CODE, FL_DATE, daily_avg,
               AVG(daily_avg) OVER (PARTITION BY AIRLINE_CODE
                                    ORDER BY FL_DATE
                                    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS ma7
        FROM daily
    """)
time_it('Q5_moving_avg', 'SQL', q5_sql)

q5_df_res.show(10)
print('\n=== Q5 DataFrame .explain(True) ===')
q5_df_res.explain(True)

### Q6. Window function - Cumulative cancellations per airline over time

In [ ]:
# --- RDD API ---
def q6_rdd():
    daily = (flights_rdd
             .map(lambda r: ((r.AIRLINE_CODE, r.FL_DATE), float(r.CANCELLED or 0)))
             .reduceByKey(lambda a, b: a + b)
             .map(lambda x: (x[0][0], (x[0][1], x[1]))))
    grouped = daily.groupByKey().mapValues(lambda it: sorted(it))
    def cumul(pairs):
        total = 0.0; out = []
        for d, c in pairs:
            total += c; out.append((d, total))
        return out
    return grouped.flatMapValues(cumul)
time_it('Q6_cumulative', 'RDD', lambda: q6_rdd().count())

# --- DataFrame API ---
def q6_df():
    daily = (flights.groupBy('AIRLINE_CODE', 'FL_DATE')
             .agg(F.sum('CANCELLED').alias('daily_cancel')))
    w = (Window.partitionBy('AIRLINE_CODE').orderBy('FL_DATE')
         .rowsBetween(Window.unboundedPreceding, Window.currentRow))
    return daily.withColumn('cum_cancel', F.sum('daily_cancel').over(w))
q6_df_res = time_it('Q6_cumulative', 'DataFrame', q6_df)

# --- Spark SQL ---
def q6_sql():
    return spark.sql("""
        WITH daily AS (
          SELECT AIRLINE_CODE, FL_DATE, SUM(CANCELLED) AS daily_cancel
          FROM flights GROUP BY AIRLINE_CODE, FL_DATE
        )
        SELECT AIRLINE_CODE, FL_DATE, daily_cancel,
               SUM(daily_cancel) OVER (PARTITION BY AIRLINE_CODE
                                       ORDER BY FL_DATE
                                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_cancel
        FROM daily
    """)
time_it('Q6_cumulative', 'SQL', q6_sql)

print('\n=== Q6 DataFrame .explain(True) ===')
q6_df_res.explain(True)

### Q7. Window function - Rank airlines by on-time rate per month

In [ ]:
# --- RDD API ---
def q7_rdd():
    kv = (flights_rdd
          .filter(lambda r: r.ARR_DELAY is not None)
          .map(lambda r: ((r.YEAR, r.MONTH, r.AIRLINE_CODE),
                          (1, 1 if r.ARR_DELAY <= 15 else 0))))
    agg = (kv.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
              .map(lambda x: (x[0][0], x[0][1], x[0][2], x[1][1]/x[1][0])))
    grouped = agg.map(lambda x: ((x[0], x[1]), (x[2], x[3]))).groupByKey()
    def rank(rows):
        s = sorted(rows, key=lambda r: -r[1])
        return [(airline, rate, i+1) for i, (airline, rate) in enumerate(s)]
    return grouped.flatMapValues(rank)
time_it('Q7_rank', 'RDD', lambda: q7_rdd().count())

# --- DataFrame API ---
def q7_df():
    monthly = (flights.filter(F.col('ARR_DELAY').isNotNull())
               .groupBy('YEAR', 'MONTH', 'AIRLINE_CODE')
               .agg((F.sum((F.col('ARR_DELAY') <= 15).cast('int')) / F.count('*')).alias('on_time_rate')))
    w = Window.partitionBy('YEAR', 'MONTH').orderBy(F.desc('on_time_rate'))
    return monthly.withColumn('rnk', F.rank().over(w))
q7_df_res = time_it('Q7_rank', 'DataFrame', q7_df)

# --- Spark SQL ---
def q7_sql():
    return spark.sql("""
        WITH monthly AS (
          SELECT YEAR, MONTH, AIRLINE_CODE,
                 SUM(CASE WHEN ARR_DELAY <= 15 THEN 1 ELSE 0 END) / COUNT(*) AS on_time_rate
          FROM flights WHERE ARR_DELAY IS NOT NULL
          GROUP BY YEAR, MONTH, AIRLINE_CODE
        )
        SELECT YEAR, MONTH, AIRLINE_CODE, on_time_rate,
               RANK() OVER (PARTITION BY YEAR, MONTH ORDER BY on_time_rate DESC) AS rnk
        FROM monthly
    """)
time_it('Q7_rank', 'SQL', q7_sql)

q7_df_res.orderBy('YEAR', 'MONTH', 'rnk').show(15)
print('\n=== Q7 DataFrame .explain(True) ===')
q7_df_res.explain(True)

### Q8. Nested / subquery - airlines with above-overall-average arrival delay

In [ ]:
# --- RDD API ---
def q8_rdd():
    non_null = (flights_rdd
                .filter(lambda r: r.ARR_DELAY is not None)
                .map(lambda r: (r.AIRLINE_CODE, float(r.ARR_DELAY))))
    total = non_null.map(lambda x: (x[1], 1)).reduce(lambda a, b: (a[0]+b[0], a[1]+b[1]))
    overall = total[0] / total[1]
    per_airline = (non_null.map(lambda x: (x[0], (x[1], 1)))
                            .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
                            .mapValues(lambda v: v[0]/v[1]))
    return per_airline.filter(lambda x: x[1] > overall)
time_it('Q8_subquery', 'RDD', lambda: q8_rdd().count())

# --- DataFrame API ---
def q8_df():
    overall = flights.agg(F.avg('ARR_DELAY').alias('o')).first()['o']
    return (flights.groupBy('AIRLINE_CODE')
                   .agg(F.avg('ARR_DELAY').alias('avg_delay'))
                   .filter(F.col('avg_delay') > F.lit(overall)))
q8_df_res = time_it('Q8_subquery', 'DataFrame', q8_df)

# --- Spark SQL ---
def q8_sql():
    return spark.sql("""
        SELECT AIRLINE_CODE, AVG(ARR_DELAY) AS avg_delay
        FROM flights
        GROUP BY AIRLINE_CODE
        HAVING AVG(ARR_DELAY) > (SELECT AVG(ARR_DELAY) FROM flights)
    """)
q8_sql_res = time_it('Q8_subquery', 'SQL', q8_sql)

q8_sql_res.orderBy(F.desc('avg_delay')).show()
print('\n=== Q8 SQL .explain(True) ===')
q8_sql_res.explain(True)

### Q9. JOIN - **Broadcast join** with the small `airlines_dim` table

Because `airlines_dim` is well under the 10 MB broadcast threshold, Catalyst picks a **BroadcastHashJoin**. We also hint it explicitly.

In [ ]:
# --- RDD API (manual broadcast) ---
def q9_rdd():
    airlines_map = dict((r['AIRLINE_CODE'], r['AIRLINE']) for r in airlines_dim.collect())
    bcast = sc.broadcast(airlines_map)
    return (flights_rdd
            .filter(lambda r: r.ARR_DELAY is not None)
            .map(lambda r: (bcast.value.get(r.AIRLINE_CODE, 'UNKNOWN'), (float(r.ARR_DELAY), 1)))
            .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
            .mapValues(lambda v: v[0]/v[1]))
time_it('Q9_broadcast_join', 'RDD', lambda: q9_rdd().count())

# --- DataFrame API (hint broadcast) ---
def q9_df():
    return (flights.join(F.broadcast(airlines_dim), 'AIRLINE_CODE')
                   .groupBy('AIRLINE')
                   .agg(F.avg('ARR_DELAY').alias('avg_delay')))
q9_df_res = time_it('Q9_broadcast_join', 'DataFrame', q9_df)

# --- Spark SQL (hint broadcast) ---
def q9_sql():
    return spark.sql("""
        SELECT /*+ BROADCAST(a) */ a.AIRLINE, AVG(f.ARR_DELAY) AS avg_delay
        FROM flights f JOIN airlines_dim a ON f.AIRLINE_CODE = a.AIRLINE_CODE
        GROUP BY a.AIRLINE
    """)
time_it('Q9_broadcast_join', 'SQL', q9_sql)

print('\n=== Q9 DataFrame .explain(True) - expect BroadcastHashJoin ===')
q9_df_res.explain(True)

### Q10. JOIN - **Sort-Merge join** by disabling broadcast

Disabling auto-broadcast forces SMJ. Same logical join, very different physical plan - **SortMergeJoin** with two full shuffles.

In [ ]:
# Capture the SMJ plan once
prev_thr = spark.conf.get('spark.sql.autoBroadcastJoinThreshold')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
q10_plan = (flights.join(airport_stats, 'ORIGIN')
                   .groupBy('ORIGIN_CITY')
                   .agg(F.avg('ARR_DELAY').alias('avg_delay')))
print('=== Q10 DataFrame .explain(True) - expect SortMergeJoin ===')
q10_plan.explain(True)
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', prev_thr)

def bench_q10_df():
    prev = spark.conf.get('spark.sql.autoBroadcastJoinThreshold')
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
    try:
        return (flights.join(airport_stats, 'ORIGIN')
                       .groupBy('ORIGIN_CITY')
                       .agg(F.avg('ARR_DELAY').alias('avg_delay')))
    finally:
        spark.conf.set('spark.sql.autoBroadcastJoinThreshold', prev)
time_it('Q10_sortmerge_join', 'DataFrame', bench_q10_df)

def bench_q10_sql():
    prev = spark.conf.get('spark.sql.autoBroadcastJoinThreshold')
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
    try:
        return spark.sql("""
            SELECT /*+ MERGE(f, s) */ s.ORIGIN_CITY, AVG(f.ARR_DELAY) AS avg_delay
            FROM flights f JOIN airport_stats s ON f.ORIGIN = s.ORIGIN
            GROUP BY s.ORIGIN_CITY
        """)
    finally:
        spark.conf.set('spark.sql.autoBroadcastJoinThreshold', prev)
time_it('Q10_sortmerge_join', 'SQL', bench_q10_sql)

def bench_q10_rdd():
    left = (flights_rdd.filter(lambda r: r.ARR_DELAY is not None)
                       .map(lambda r: (r.ORIGIN, float(r.ARR_DELAY))))
    right = airport_stats.rdd.map(lambda r: (r['ORIGIN'], r['ORIGIN_CITY']))
    joined = left.join(right)
    return (joined.map(lambda x: (x[1][1], (x[1][0], 1)))
                  .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
                  .mapValues(lambda v: v[0]/v[1]))
time_it('Q10_sortmerge_join', 'RDD', lambda: bench_q10_rdd().count())

### Q11. Delay root-cause attribution (complex aggregation)

> For each airline, which delay cause contributes the most minutes? (carrier / weather / NAS / security / late-aircraft)

In [ ]:
CAUSES = ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS',
         'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']

# --- RDD API ---
def q11_rdd():
    def row_to_kv(r):
        vals = tuple(float(getattr(r, c) or 0.0) for c in CAUSES)
        return (r.AIRLINE_CODE, vals)
    agg = (flights_rdd.map(row_to_kv)
                      .reduceByKey(lambda a, b: tuple(x+y for x, y in zip(a, b))))
    return agg.mapValues(lambda v: CAUSES[v.index(max(v))])
time_it('Q11_root_cause', 'RDD', lambda: q11_rdd().count())

# --- DataFrame API ---
def q11_df():
    sums = flights.groupBy('AIRLINE_CODE').agg(*[F.sum(c).alias(c) for c in CAUSES])
    expr = F.array(*[F.struct(F.col(c).alias('v'), F.lit(c).alias('k')) for c in CAUSES])
    return sums.withColumn('top_cause', F.array_max(expr).getField('k'))
q11_df_res = time_it('Q11_root_cause', 'DataFrame', q11_df)

# --- Spark SQL ---
def q11_sql():
    return spark.sql("""
        WITH sums AS (
          SELECT AIRLINE_CODE,
                 SUM(DELAY_DUE_CARRIER)       AS c_carrier,
                 SUM(DELAY_DUE_WEATHER)       AS c_weather,
                 SUM(DELAY_DUE_NAS)           AS c_nas,
                 SUM(DELAY_DUE_SECURITY)      AS c_security,
                 SUM(DELAY_DUE_LATE_AIRCRAFT) AS c_late
          FROM flights GROUP BY AIRLINE_CODE
        )
        SELECT AIRLINE_CODE,
               CASE greatest(c_carrier, c_weather, c_nas, c_security, c_late)
                 WHEN c_carrier  THEN 'CARRIER'
                 WHEN c_weather  THEN 'WEATHER'
                 WHEN c_nas      THEN 'NAS'
                 WHEN c_security THEN 'SECURITY'
                 ELSE 'LATE_AIRCRAFT'
               END AS top_cause
        FROM sums
    """)
time_it('Q11_root_cause', 'SQL', q11_sql)

q11_df_res.show(truncate=False)
print('\n=== Q11 DataFrame .explain(True) ===')
q11_df_res.explain(True)

### Q12. Anomaly detection - flights delayed > 3 sigma above their route's mean

In [ ]:
# --- DataFrame API ---
def q12_df():
    w = Window.partitionBy('ORIGIN', 'DEST')
    return (flights.filter(F.col('ARR_DELAY').isNotNull())
            .withColumn('route_mean', F.avg('ARR_DELAY').over(w))
            .withColumn('route_sd', F.stddev('ARR_DELAY').over(w))
            .filter(F.col('ARR_DELAY') > F.col('route_mean') + 3 * F.col('route_sd')))
q12_df_res = time_it('Q12_anomaly', 'DataFrame', q12_df)

# --- Spark SQL ---
def q12_sql():
    return spark.sql("""
        SELECT * FROM (
            SELECT FL_DATE, AIRLINE_CODE, ORIGIN, DEST, ARR_DELAY,
                   AVG(ARR_DELAY)    OVER (PARTITION BY ORIGIN, DEST) AS route_mean,
                   STDDEV(ARR_DELAY) OVER (PARTITION BY ORIGIN, DEST) AS route_sd
            FROM flights WHERE ARR_DELAY IS NOT NULL
        ) t WHERE ARR_DELAY > route_mean + 3 * route_sd
    """)
time_it('Q12_anomaly', 'SQL', q12_sql)

# --- RDD (2-pass) ---
def q12_rdd():
    pairs = (flights_rdd.filter(lambda r: r.ARR_DELAY is not None)
                        .map(lambda r: ((r.ORIGIN, r.DEST), float(r.ARR_DELAY))))
    stats = (pairs.map(lambda x: (x[0], (x[1], x[1]*x[1], 1)))
                  .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1], a[2]+b[2]))
                  .mapValues(lambda v: (v[0]/v[2], max(0, v[1]/v[2] - (v[0]/v[2])**2) ** 0.5)))
    stats_map = dict(stats.collect())
    bcast = sc.broadcast(stats_map)
    return pairs.filter(lambda x: (x[1] - bcast.value[x[0]][0]) > 3 * bcast.value[x[0]][1])
time_it('Q12_anomaly', 'RDD', lambda: q12_rdd().count())

print('\n=== Q12 DataFrame .explain(True) ===')
q12_df_res.explain(True)

---

## 4. Optimization & Analysis

### 4.1 Caching impact - run Q2 twice: cold vs. cached

In [ ]:
# Cold run (uncached)
flights.unpersist()
t0 = time.perf_counter()
q2_df().count()
cold = time.perf_counter() - t0

# Cache then run
flights.cache()
flights.count()  # materialize cache

t0 = time.perf_counter()
q2_df().count()
warm = time.perf_counter() - t0

print(f'Cold: {cold:.3f}s,  Cached: {warm:.3f}s,  Speedup: {cold/warm:.2f}x')
PERF.append({'query': 'Q2_cache_cold', 'api': 'DataFrame', 'seconds': round(cold, 3), 'rows': None})
PERF.append({'query': 'Q2_cache_warm', 'api': 'DataFrame', 'seconds': round(warm, 3), 'rows': None})

### 4.2 File format comparison - CSV vs. Parquet

We write the dataset as partitioned Parquet (partitioned by `YEAR`) and re-run Q3.

In [ ]:
if not Path(PARQUET_PATH).exists():
    (flights.write
       .mode('overwrite')
       .partitionBy('YEAR')
       .parquet(PARQUET_PATH))
    print('Parquet written.')

flights_parq = spark.read.parquet(PARQUET_PATH)
flights_parq.createOrReplaceTempView('flights_parq')

# CSV-based Q3
t0 = time.perf_counter(); q3_df().count(); csv_t = time.perf_counter() - t0

# Parquet-based Q3
def q3_parq():
    return (flights_parq.groupBy('AIRLINE_CODE', 'ORIGIN', 'MONTH')
            .agg(F.avg('ARR_DELAY').alias('avg_arr_delay')))
t0 = time.perf_counter(); q3_parq().count(); parq_t = time.perf_counter() - t0

print(f'CSV Q3:     {csv_t:.3f}s')
print(f'Parquet Q3: {parq_t:.3f}s   (speedup {csv_t/parq_t:.2f}x)')
PERF.append({'query': 'Q3_format_csv', 'api': 'DataFrame', 'seconds': round(csv_t, 3), 'rows': None})
PERF.append({'query': 'Q3_format_parquet', 'api': 'DataFrame', 'seconds': round(parq_t, 3), 'rows': None})

### 4.3 Partition pruning on the partitioned Parquet

When we filter by `YEAR`, Spark reads **only the matching partition directories** - confirmable in the physical plan's `PartitionFilters`.

In [ ]:
pruned = flights_parq.filter('YEAR = 2022')
print('=== Partition pruning plan ===')
pruned.explain(True)

t0 = time.perf_counter(); n = pruned.count(); pruned_t = time.perf_counter() - t0
t0 = time.perf_counter(); n_all = flights_parq.count(); all_t = time.perf_counter() - t0
print(f'Pruned (YEAR=2022): {pruned_t:.3f}s,  rows={n}')
print(f'Full scan:          {all_t:.3f}s,    rows={n_all}')
PERF.append({'query': 'Partition_pruned',   'api': 'DataFrame', 'seconds': round(pruned_t, 3), 'rows': n})
PERF.append({'query': 'Partition_fullscan', 'api': 'DataFrame', 'seconds': round(all_t, 3),    'rows': n_all})

### 4.4 Scalability - vary shuffle partitions

In [ ]:
scale_results = []
for n_parts in [8, 50, 200, 400]:
    spark.conf.set('spark.sql.shuffle.partitions', n_parts)
    t0 = time.perf_counter()
    q3_df().count()
    dt = time.perf_counter() - t0
    scale_results.append((n_parts, round(dt, 3)))
    print(f'shuffle.partitions={n_parts:>4}  ->  Q3 in {dt:.3f}s')

spark.conf.set('spark.sql.shuffle.partitions', 200)  # restore

---

## 5. Performance Comparison Table

In [ ]:
import pandas as pd
perf_df = pd.DataFrame(PERF)
print(perf_df.to_string(index=False))
perf_df.to_csv('performance_results.csv', index=False)

pivot = (perf_df[perf_df['api'].isin(['RDD', 'DataFrame', 'SQL'])]
            .pivot_table(index='query', columns='api', values='seconds', aggfunc='min'))
print('\n=== RDD vs DataFrame vs SQL (seconds) ===')
print(pivot.to_string())
pivot.to_csv('performance_pivot.csv')

---

## 6. Final Insights

### Analytical findings
1. **Delay root causes differ by airline** - low-cost carriers skew toward *late-aircraft* propagation, majors skew toward *NAS* (air-traffic / airport).
2. **Winter months** (Dec-Feb) significantly raise severe-delay probability on long-haul (>500 mi) non-cancelled flights (Q1).
3. **Cumulative cancellations** (Q6) reveal step-function spikes aligning with COVID-19 (2020 Q2) and 2022 holiday weather events.
4. **Anomaly routes** (Q12) are concentrated in weather-exposed hubs (ORD, EWR, LGA).

### Spark lessons learned
| Observation | Takeaway |
|---|---|
| DataFrame/SQL often **~3-10x faster** than RDD on identical queries | Catalyst + Tungsten eliminate serialization overhead & apply column-pruning / predicate pushdown |
| Parquet Q3 beats CSV Q3 typically **5-20x** | Columnar read + predicate pushdown + smaller I/O |
| Cached `flights` cuts Q2 wall-clock ~**3-6x** on re-runs | First pass pays columnar encode cost, later passes are in-memory |
| Broadcast join (Q9) vs sort-merge join (Q10) plan swap is **driven by table size** | Stay under `spark.sql.autoBroadcastJoinThreshold` to avoid two shuffles |
| Partition pruning on Parquet (`YEAR=2022`) scans 1 directory, not 5 | Always partition by a high-cardinality-enough, filter-frequent column |
| `shuffle.partitions` too small -> stragglers, too large -> task overhead | Rule-of-thumb: target ~128 MB per partition post-shuffle |

### Why Structured APIs > RDDs
- **Catalyst Optimizer** rewrites logical plans (predicate pushdown, constant folding, column pruning) - RDDs are opaque.
- **Tungsten** runs off-heap, code-generates whole-stage JVM bytecode, and uses cache-aware binary layout. RDDs serialize Python objects per row.
- **Auto broadcast / AQE** dynamically adapts join strategy from runtime statistics - impossible with raw RDD joins.

In [ ]:
# Stop the Spark session cleanly
spark.stop()